# NB05 — Training Only (T4 Test)Minimal notebook: clone → install → copy data → **train**.No simulation, no drift, no Airflow. Just GPU training.**Purpose:** Verify training completes on T4 before running full 40-day simulation.**Expected:** ~5 hrs on T4 (early stop ~epoch 33, ~530s/epoch)**Fixes in this version:**- GradScaler `init_scale=1024` (was 65536 — caused float16 overflow with FocalLoss)- NaN batches are **skipped** instead of crashing (GradScaler adapts scale down)- Only crashes if >50% of batches in an epoch are NaN

In [ ]:
import os, shutil, subprocessREPO = "https://github.com/AIML-Engineering-Lab/053_dram_memory_yield_mlops.git"DIR = "/content/053_memory_yield_predictor"try:    from google.colab import userdata    tok = userdata.get('GITHUB_TOKEN')except Exception:    tok = Noneurl = REPO.replace("https://", f"https://{tok}@") if tok else REPOif os.path.exists(DIR):    shutil.rmtree(DIR)r = subprocess.run(["git", "clone", url, DIR], capture_output=True, text=True)if r.returncode != 0:    raise RuntimeError(f"Clone failed: {r.stderr}")os.chdir(DIR)# Verify NaN fix is presentwith open('src/train.py') as f:    src = f.read()assert 'init_scale=1024' in src, 'GradScaler init_scale fix MISSING! Push latest code.'assert 'nan_batches' in src, 'NaN recovery fix MISSING! Push latest code.'print('\u2705 Cloned + NaN fixes verified')subprocess.run(["pip", "install", "-q", "mlflow", "boto3", "pyarrow", "pandas",                "numpy", "scikit-learn", "xgboost", "lightgbm"],               capture_output=True)print('\u2705 Deps installed')

In [ ]:
import torchfrom google.colab import drivefrom pathlib import Pathimport shutilif not torch.cuda.is_available():    raise RuntimeError("No GPU! Runtime > Change runtime type > T4 GPU")gpu = torch.cuda.get_device_name(0)cc = torch.cuda.get_device_capability(0)vram = torch.cuda.get_device_properties(0).total_memory / 1e9bs = 4096 if cc[0] >= 8 else 2048amp = 'bfloat16' if cc[0] >= 8 else 'float16+GradScaler(init_scale=1024)'print(f'\u2705 {gpu} ({vram:.1f} GB, cc {cc[0]}.{cc[1]})')print(f'   batch_size={bs}, {amp}')drive.mount('/content/drive')dest = '/content/053_memory_yield_predictor/data/preprocessed_full.npz'Path(dest).parent.mkdir(parents=True, exist_ok=True)for src in ['/content/drive/MyDrive/P053_data/preprocessed_full.npz',            '/content/drive/MyDrive/preprocessed_full.npz']:    if Path(src).exists():        shutil.copy2(src, dest)        mb = Path(dest).stat().st_size / 1e6        print(f'\u2705 Copied {mb:.0f} MB from {src}')        assert mb > 2000, f'File too small ({mb:.0f} MB)'        breakelse:    raise FileNotFoundError('preprocessed_full.npz not found on Drive')

In [ ]:
import subprocess, sys, os, timeDIR = '/content/053_memory_yield_predictor'os.chdir(DIR)# Detect batch_size from GPUimport torchcc = torch.cuda.get_device_capability(0)[0]bs = 4096 if cc >= 8 else 2048cmd = [sys.executable, '-u', '-m', 'src.train',       '--full', '--epochs', '50', '--batch-size', str(bs),       '--run-name', 'colab-t4-v7A-test', '--context', 'colab']print(f'Command: {" ".join(cmd)}')print(f'Expected: ~5 hrs on T4 (early stop ~epoch 33, ~530s/epoch)')print('=' * 70, flush=True)env = os.environ.copy()env['PYTHONUNBUFFERED'] = '1't0 = time.time()proc = subprocess.Popen(    cmd, cwd=DIR, env=env,    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,    text=True, bufsize=1,)for line in proc.stdout:    print(line, end='', flush=True)retcode = proc.wait()elapsed = (time.time() - t0) / 60print(f'\n{"=" * 70}')if retcode == 0:    print(f'\u2705 Training complete in {elapsed:.1f} min')else:    print(f'\u274c Training FAILED (exit {retcode}) after {elapsed:.1f} min')print('=' * 70)

In [ ]:
import jsonfrom pathlib import PathDIR = '/content/053_memory_yield_predictor'for b in sorted(Path(f'{DIR}/data').glob('benchmark_*.json'),                key=lambda f: f.stat().st_mtime, reverse=True):    bm = json.loads(b.read_text())    print(f'Benchmark: {b.name}')    print(f'  GPU: {bm.get("gpu_name")} | batch: {bm.get("batch_size")}')    print(f'  Epochs: {bm.get("epochs_run")} (best: {bm.get("best_epoch")})')    print(f'  Time: {bm.get("total_train_time_min", 0):.1f} min')    print(f'  Avg epoch: {bm.get("avg_epoch_time_s", 0):.1f}s')    print(f'  Peak VRAM: {bm.get("peak_gpu_memory_gb", 0):.1f} GB')    if 'results' in bm and 'val' in bm['results']:        v = bm['results']['val']        print(f'  Val AUC-PR: {v["auc_pr"]:.4f} | F1: {v["f1"]:.4f}')    breakarts = list(Path(f'{DIR}/src/artifacts').glob('*.pt'))if arts:    print(f'\n\u2705 Model: {arts[0].name} ({arts[0].stat().st_size/1e6:.1f} MB)')else:    print('\n\u26a0\ufe0f No model .pt found')# Copy to Driveimport shutildrive_dir = Path('/content/drive/MyDrive/P053_results/training_test')drive_dir.mkdir(parents=True, exist_ok=True)for f in Path(f'{DIR}/data').glob('benchmark_*.json'):    shutil.copy2(f, drive_dir / f.name)for f in Path(f'{DIR}/src/artifacts').glob('*.pt'):    shutil.copy2(f, drive_dir / f.name)print(f'\n\u2705 Results copied to {drive_dir}')